
# Python Inference Tutorial - Single Model

This tutorial describes how to run an inference process using `InferPipeline` API (sync API), which is an alternative to the recommended Async API


**Requirements:**

* Run the notebook inside the Python virtual environment: ```source hailo_virtualenv/bin/activate```

When inside the ```virtualenv```, use the command ``hailo tutorial`` to open a Jupyter server that contains the tutorials.

In [2]:
!pwd

/home/trap-fish/uav-human-detection/hailo-ai/notebooks


In [1]:
import numpy as np
from multiprocessing import Process
from hailo_platform import (HEF, VDevice, HailoStreamInterface, InferVStreams, ConfigureParams,
    InputVStreamParams, OutputVStreamParams, InputVStreams, OutputVStreams, FormatType)

# The target can be used as a context manager ("with" statement) to ensure it's released on time.
# Here it's avoided for the sake of simplicity
target = VDevice()

# Loading compiled HEFs to device:
# model_name = 'yolo11n'
# hef_path = '../hefs/{}.hef'.format(model_name) 
hef_path = '/home/trap-fish/uav-human-detection/models/yolo11n_visdrone.hef'
hef = HEF(hef_path)
    
# Configure network groups
configure_params = ConfigureParams.create_from_hef(hef=hef, interface=HailoStreamInterface.PCIe)
network_groups = target.configure(hef, configure_params)
network_group = network_groups[0]
network_group_params = network_group.create_params()

# Create input and output virtual streams params
input_vstreams_params = InputVStreamParams.make(network_group, format_type=FormatType.FLOAT32)
output_vstreams_params = OutputVStreamParams.make(network_group, format_type=FormatType.FLOAT32)

# Define dataset params
input_vstream_info = hef.get_input_vstream_infos()[0]
output_vstream_info = hef.get_output_vstream_infos()[0]
image_height, image_width, channels = input_vstream_info.shape
num_of_images = 10
low, high = 2, 20

# Generate random dataset
dataset = np.random.randint(low, high, (num_of_images, image_height, image_width, channels)).astype(np.float32)

#### Running hardware inference
Infer the model and then display the output shape:

In [3]:
# Infer 
with InferVStreams(network_group, input_vstreams_params, output_vstreams_params) as infer_pipeline:
    input_data = {input_vstream_info.name: dataset}
    with network_group.activate(network_group_params):
        infer_results = infer_pipeline.infer(input_data)
        print(f"Stream output shape is {infer_results[output_vstream_info.name][0][:]}")

infer_results

Stream output shape is [array([], shape=(0, 5), dtype=float64), array([], shape=(0, 5), dtype=float64)]


{'yolo11n_visdrone/yolov8_nms_postprocess': [[array([], shape=(0, 5), dtype=float64),
   array([], shape=(0, 5), dtype=float64)],
  [array([], shape=(0, 5), dtype=float64),
   array([], shape=(0, 5), dtype=float64)],
  [array([], shape=(0, 5), dtype=float64),
   array([], shape=(0, 5), dtype=float64)],
  [array([], shape=(0, 5), dtype=float64),
   array([], shape=(0, 5), dtype=float64)],
  [array([], shape=(0, 5), dtype=float64),
   array([], shape=(0, 5), dtype=float64)],
  [array([], shape=(0, 5), dtype=float64),
   array([], shape=(0, 5), dtype=float64)],
  [array([], shape=(0, 5), dtype=float64),
   array([], shape=(0, 5), dtype=float64)],
  [array([], shape=(0, 5), dtype=float64),
   array([], shape=(0, 5), dtype=float64)],
  [array([], shape=(0, 5), dtype=float64),
   array([], shape=(0, 5), dtype=float64)],
  [array([], shape=(0, 5), dtype=float64),
   array([], shape=(0, 5), dtype=float64)]]}

## Streaming inference

This section shows how to run streaming inference using multiple processes in Python.

Note: This flow is not supported on Windows.

We will not use infer. Instead we will use a send and receive model.
The send function and the receive function will run in different processes.

Define the send and receive functions:

In [ ]:
def send(configured_network, num_frames):
    configured_network.wait_for_activation(1000)
    vstreams_params = InputVStreamParams.make(configured_network)
    with InputVStreams(configured_network, vstreams_params) as vstreams:
        vstream_to_buffer = {vstream: np.ndarray([1] + list(vstream.shape), dtype=vstream.dtype) for vstream in vstreams}
        for _ in range(num_frames):
            for vstream, buff in vstream_to_buffer.items():
                vstream.send(buff)

def recv(configured_network, vstreams_params, num_frames):
    configured_network.wait_for_activation(1000)
    with OutputVStreams(configured_network, vstreams_params) as vstreams:
        for _ in range(num_frames):
            for vstream in vstreams:
                data = vstream.recv()

def recv_all(configured_network, num_frames):
    vstreams_params_groups = OutputVStreamParams.make_groups(configured_network)
    recv_procs = []
    for vstreams_params in vstreams_params_groups:
        proc = Process(target=recv, args=(configured_network, vstreams_params, num_frames))
        proc.start()
        recv_procs.append(proc)
    for proc in recv_procs:
        proc.join()

Define the amount of frames to stream, define the processes, create the target and run processes:


In [ ]:
# Define the amount of frames to stream
num_of_frames = 1000
model_name = "yolo11n_visdrone"

send_process = Process(target=send, args=(network_group, num_of_frames))
recv_process = Process(target=recv_all, args=(network_group, num_of_frames))
recv_process.start()
send_process.start()
print('Starting streaming (hef=\'{}\', num_of_frames={})'.format(model_name, num_of_frames))
with network_group.activate(network_group_params):
    send_process.join()
    recv_process.join()
print('Done')

target.release()

## CJM's Tutorial on Hailo8
Seems to mostly just be the above code with minor adjustments

In [1]:


# OpenCV for computer vision tasks
import cv2

# NumPy for numerical operations
import numpy as np

# PIL (Python Imaging Library) for image processing
from PIL import Image, ImageDraw, ImageFont

# Import Hailo Runtime dependencies
from hailo_platform import (
    HEF,
    ConfigureParams,
    FormatType,
    HailoSchedulingAlgorithm,
    HailoStreamInterface,
    InferVStreams,
    InputVStreamParams,
    OutputVStreamParams,
    VDevice
)

# Import Picamera2
# from picamera2 import Picamera2, Preview

In [2]:

# Load the compiled HEF to Hailo device
hef_path = '/home/trap-fish/uav-human-detection/models/yolo11n_visdrone.hef'
hef = HEF(str(hef_path))

# Set VDevice (Virtual Device) params to disable the HailoRT service feature
params = VDevice.create_params()
params.scheduling_algorithm = HailoSchedulingAlgorithm.NONE

# Create a Hailo virtual device with the specified parameters
target = VDevice(params=params)

# Get the "network groups" (connectivity groups, aka. "different networks") information from the .hef
# Configure the device with the HEF and PCIe interface
configure_params = ConfigureParams.create_from_hef(hef=hef, interface=HailoStreamInterface.PCIe)
network_groups = target.configure(hef, configure_params)

# Select the first network group (there's only one in this case)
network_group = network_groups[0]
network_group_params = network_group.create_params()

# Create input and output virtual streams params
# These specify the format of the input and output data (in this case, 32-bit float)
input_vstreams_params = InputVStreamParams.make(network_group, format_type=FormatType.FLOAT32)
output_vstreams_params = OutputVStreamParams.make(network_group, format_type=FormatType.FLOAT32)

# Get information about the input and output virtual streams
input_vstream_info = hef.get_input_vstream_infos()[0]
output_vstream_info = hef.get_output_vstream_infos()[0]

In [ ]:
!pip install torchvision

In [3]:
import torchvision as tv

def preproc(image, output_height=640, output_width=640):
    preprocess = tv.transforms.Compose([
        tv.transforms.Resize((output_height, output_width)),
    ])
    
    data = np.array(preprocess(image))
    
    return data

In [4]:

sample_frame_path = "/home/trap-fish/uav-human-detection/hailo-ai/exported_from_docker/val/images/0000001_03999_d_0000007.jpg"
# Capture a frame
frame = Image.open(sample_frame_path)

input_img_np = preproc(frame)

# Resize and pad the sample image to the desired input size, retrieving transformation data.
#input_img_np, transform_data = resize_and_pad(frame, input_vstream_info.shape[::-1][1:], True)

# Convert the input image to NumPy format for the model
input_tensor_np = np.array(input_img_np, dtype=np.float32)[None]
input_tensor_np = np.ascontiguousarray(input_tensor_np)

# Run inference
input_data = {input_vstream_info.name: input_tensor_np}
with InferVStreams(network_group, input_vstreams_params, output_vstreams_params) as infer_pipeline:
    with network_group.activate(network_group_params):
        infer_results = infer_pipeline.infer(input_data)

# Transpose and extract the first element of the quantized results
outputs = infer_results[output_vstream_info.name]#.transpose(0, 1, 3, 2)[0]


In [5]:
def run_inference(vstream_params, input_data):
    network_group, input_vstreams_params, output_vstreams_params = vstream_params
    #input_data = {input_vstream_info.name: input_tensor}
    with InferVStreams(network_group, input_vstreams_params, output_vstreams_params) as infer_pipeline:
        with network_group.activate(network_group_params):
            infer_results = infer_pipeline.infer(input_data)
    return infer_results

In [6]:
import json
import os
import time
annotations_file = '/home/trap-fish/uav-human-detection/hailo-ai/exported_from_docker/annotations_VisDroneHumans_val.json'
images_path = "/home/trap-fish/uav-human-detection/hailo-ai/exported_from_docker/VisDrone2019-DET-val/images"

with open(annotations_file, 'r') as f:
    val_gt = json.load(f)
    f.close()
    
images = val_gt['images']
image_list = [(x['file_name'],x['id']) for x in images] 

print(f"frame count: {len(image_list)}")

frame count: 532


In [7]:
dataset_sz = len(image_list)
val_dataset = np.zeros((dataset_sz, 640, 640, 3))
for idx, imagename_id in enumerate(image_list):
    imgname, imgid = imagename_id
    image_file = os.path.join(images_path, imgname)
    img = Image.open(image_file).convert('RGB')
    img_preproc = preproc(img)
    val_dataset[idx, :, :, :] = img_preproc

In [8]:
# create an np array for the validation dataset
vstream_config = (network_group, input_vstreams_params, output_vstreams_params)
outs = []
start_time = time.perf_counter()
for idx, imagename_id in enumerate(image_list):
    img_arr = val_dataset[idx, :, :, :]
    
    # # Convert the input image to NumPy format for the model
    input_tensor_np = np.array(img_arr, dtype=np.float32)[None]
    input_tensor_np = np.ascontiguousarray(input_tensor_np)
    
    input_data = {input_vstream_info.name: input_tensor_np}

    outs.append(run_inference(vstream_config, input_data))

# Calculate and display FPS
end_time = time.perf_counter()
processing_time = end_time - start_time
fps = 1 / processing_time


In [9]:
class0preds = []
class1preds = []
for idx, el in enumerate(outs):
    class0preds.append(el.get('yolo11n_visdrone/yolov8_nms_postprocess')[0][0])
    class1preds.append(el.get('yolo11n_visdrone/yolov8_nms_postprocess')[0][1])


In [16]:
outs

[{'yolo11n_visdrone/yolov8_nms_postprocess': [[array([[0.33522993, 0.628282  , 0.36738434, 0.6405525 , 0.593077  ],
           [0.21119586, 0.55237705, 0.2427885 , 0.5601248 , 0.5503385 ],
           [0.16610898, 0.54863554, 0.1912871 , 0.5591285 , 0.51906925],
           [0.33314267, 0.6177434 , 0.36910802, 0.6299912 , 0.4158808 ],
           [0.27370492, 0.66245717, 0.30673856, 0.6750475 , 0.4065    ],
           [0.11611014, 0.5552583 , 0.1387844 , 0.56498367, 0.39399233],
           [0.16243723, 0.54089373, 0.18430081, 0.5488268 , 0.24077308]],
          dtype=float32),
    array([], shape=(0, 5), dtype=float64)]]},
 {'yolo11n_visdrone/yolov8_nms_postprocess': [[array([[0.42454523, 0.301214  , 0.45248508, 0.3112797 , 0.68166924],
           [0.9749689 , 0.4598756 , 1.00005   , 0.47259888, 0.36585003],
           [0.98875153, 0.51489973, 1.0000278 , 0.52861184, 0.29705772],
           [0.6750407 , 0.34230804, 0.7014043 , 0.35246843, 0.26266155],
           [0.2094552 , 0.32116747, 0

In [12]:

# Process the model output to get object proposals
proposals = process_outputs(outputs, input_vstream_info.shape[:-1], bbox_conf_thresh)

# Apply non-max suppression to filter overlapping proposals
proposal_indices = nms_sorted_boxes(calc_iou(proposals[:, :-2]), iou_thresh)
proposals = proposals[proposal_indices]

# Extract bounding boxes, labels, and probabilities from proposals
bbox_list = [adjust_bbox(bbox, transform_data) for bbox in proposals[:,:4]]
label_list = [class_names[int(idx)] for idx in proposals[:,4]]
probs_list = proposals[:,5]

# Initialize track IDs for detected objects
track_ids = [-1]*len(bbox_list)

# Convert bounding boxes to top-left bottom-right (tlbr) format
tlbr_boxes = np.array(bbox_list).reshape(-1,4).copy()
tlbr_boxes[:, 2:4] += tlbr_boxes[:, :2]

# Update tracker with detections
tracks = tracker.update(
    output_results=np.concatenate([tlbr_boxes, probs_list[:, np.newaxis]], axis=1),
    img_info=frame.shape[::-1][1:],
    img_size=frame.shape[::-1][1:])

if len(tlbr_boxes) > 0 and len(tracks) > 0:
    # Match detections with tracks
    track_ids = match_detections_with_tracks(tlbr_boxes=tlbr_boxes, track_ids=track_ids, tracks=tracks)

    # Filter object detections based on tracking results
    bbox_list, label_list, probs_list, track_ids = zip(*[(bbox, label, prob, track_id) 
                                                        for bbox, label, prob, track_id 
                                                        in zip(bbox_list, label_list, probs_list, track_ids) if track_id != -1])
    
    if len(bbox_list) > 0:
        # Annotate the current frame with bounding boxes and tracking IDs
        annotated_img = draw_bboxes_pil(
            image=Image.fromarray(frame), 
            boxes=bbox_list, 
            labels=[f"{track_id}-{label}" for track_id, label in zip(track_ids, label_list)],
            probs=probs_list,
            colors=[int_colors[class_names.index(i)] for i in label_list],  
            font=font_file,
        )
        annotated_frame = cv2.resize(np.array(annotated_img), (preview_width, preview_height))
else:
    # If no detections, use the original frame
    annotated_frame = cv2.resize(frame, (preview_width, preview_height))

# Calculate and display FPS
end_time = time.perf_counter()
processing_time = end_time - start_time
fps = 1 / processing_time

fps_text = f"FPS: {fps:.2f}"
cv2.putText(annotated_frame, fps_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

# Display the annotated frame
cv2.imshow(window_title, annotated_frame)

# Check for 'q' key press to exit the loop
if cv2.waitKey(1) & 0xFF == ord('q'):
    break
finally:
    # Stop the camera and close the preview window
    cv2.destroyAllWindows()
    picam2.close()

SyntaxError: invalid syntax (4164154174.py, line 64)